# Notebook 21: Matched ACE Intervention & 20-Direction Orthogonal Hypersphere Null (Colab GPU)

**Protocol Identifier:** `ACE-MATCHED-ARR2027-HYPERSPHERE-NULL`  
**Governing Document:** ARR 2027 Remediation Directive (Item A.6)  
**Target Venue:** ACL 2027 (ARR Cycle)  
**Hardware Target:** Google Colab A100 GPU (~17.9 Compute Units)  

### Scientific Objectives:
1. Address Reviewer Objection 7: The original Attractor-Contrast-Escape (ACE) intervention reported by Zhang et al. (2026) shifted Generative Perplexity and per-sample entropy, confounding steering efficacy with operating-point drift.
2. Construct a **20-direction random orthogonal hypersphere null distribution** on $\mathbb{S}^{510} \subset \mathbb{R}^{512}$:
   - In ELF-B, self-conditioning feedback resides in the hidden state space of dimension $d = 512$.
   - Sample 20 random unit directions strictly orthogonal to $\hat{\mathbf{d}}_{\mathrm{ACE}}$ ($|\langle \mathbf{v}_{\perp, k}, \hat{\mathbf{d}}_{\mathrm{ACE}} \rangle| < 10^{-6}$).
   - Evaluate all 20 null directions across held-out seeds to prove that repetition reduction is not an artifact of generic subspace projection.
3. Conduct **published-scale ACE calibration (4,096 trajectories across seeds 301–305)**:
   - Roll out trajectories under $(\mathrm{SC}=4.0, \gamma=1.000)$ with `sweep.HOOKS["ace_record"]` capturing mean self-conditioning feedback $\mathbf{u}_j \in \mathbb{R}^{512}$.
   - Contrast top 25% highest-rep4 and lowest-rep4 quartiles.
   - Enforce cosine stability check: $|\langle \hat{\mathbf{d}}_{\mathrm{ACE}}^{(4096)}, \hat{\mathbf{d}}_{\mathrm{ACE}}^{(500)} \rangle| \ge 0.85$.
4. Implement **$\gamma$-compensated matched ACE calibration**:
   - Calibrate the compensating Langevin back-noising parameter $\gamma_{\mathrm{comp}}^*$ on 2 calibration seeds (301, 302) under ACE projection ($\alpha=0.5, \mathrm{SC}=4.0$) to restore baseline Gen-PPL ($27.50$ nats).
   - Evaluate on 5 held-out evaluation seeds (201–205) under formal paired Joint TOST against unsteered baseline $(\mathrm{SC}=1.0, \gamma^*=1.651)$.
   - Measure residual repetition reduction at the true matched operating point.


In [ ]:
# Cell 1: Fast Google Colab Environment Setup, Drive Mount, & Dependency Check
import datetime
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch

# ==============================================================================
# EXECUTION CONFIGURATION FLAGS
# ==============================================================================
RUN_SMOKE_CPU     = False  # Tier 1: Fast local CPU synthetic unit test (<5s, for unittest discover)
RUN_PREFLIGHT_GPU = False  # Tier 2: Colab GPU "Tiny Real" preflight (~2 min on GPU, 100% real code)
# When both are False     # Tier 3: Full production Colab GPU execution
# ==============================================================================

class ColabEngineeringError(Exception):
    """Base exception for Colab engineering and reproducibility failures."""
    pass

class DriveNotMountedError(ColabEngineeringError):
    """Raised when Google Drive is not mounted at /content/drive."""
    pass

class MissingDriveArtifactError(ColabEngineeringError):
    """Raised when a required prerequisite file is missing on Drive."""
    pass

def assert_colab_drive_mounted() -> Path:
    if RUN_SMOKE_CPU:
        return Path.cwd()
    if "google.colab" not in sys.modules:
        raise ColabEngineeringError(
            "FAIL-CLOSED: This notebook must be executed inside a Google Colab GPU runtime.\n"
            "To test algorithmic logic locally on CPU, set RUN_SMOKE_CPU = True."
        )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/elf_research")
    if not root.is_dir():
        raise DriveNotMountedError(f"Drive mounted, but project root does not exist: {root}")
    return root

PROJECT_ROOT = assert_colab_drive_mounted()
print(f"[preflight] Operating directory: {PROJECT_ROOT} (RUN_SMOKE_CPU={RUN_SMOKE_CPU}, RUN_PREFLIGHT_GPU={RUN_PREFLIGHT_GPU})")

if not RUN_SMOKE_CPU:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [CODE_DIR, RUNS_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    # 1. Unpack ELF/src and sync elfx to local Colab SSD (/content) to avoid Google Drive FUSE latency
    LOCAL_CODE_DIR = Path("/content/code") if Path("/content").is_dir() else CODE_DIR
    if torch.cuda.is_available():
        if not os.path.isdir("/content/repo/ELF/src"):
            zip_candidates = [CODE_DIR / "elf_code.zip", PROJECT_ROOT / "elf_code.zip", Path("/content/elf_code.zip")]
            found_zip = next((p for p in zip_candidates if p.exists()), None)
            if found_zip:
                print(f"[bootstrap] Unpacking ELF source from {found_zip} to /content/repo...")
                subprocess.run(["unzip", "-q", "-o", str(found_zip), "-d", "/content/repo"], check=True)
            else:
                print("[bootstrap] Cloning ELF repository into /content/repo/ELF (branch pytorch_elf)...")
                os.makedirs("/content/repo", exist_ok=True)
                subprocess.run([
                    "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                    "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
                ], check=True)

        if (CODE_DIR / "elfx").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)
        if (PROJECT_ROOT / "frontier_audit").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            shutil.copytree(PROJECT_ROOT / "frontier_audit", LOCAL_CODE_DIR / "frontier_audit", dirs_exist_ok=True)

    # 2. Configure environment variables and sys.path (prioritizing fast local SSD)
    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR), str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 3. Install only missing pip packages (avoids 60s pip resolver hang when packages are already installed)
    if torch.cuda.is_available():
        _REQ_SPECS = {"mauve": "mauve-text>=0.3.0", "sacrebleu": "sacrebleu>=2.4.0", "rouge_score": "rouge-score>=0.1.2", "einops": "einops>=0.7.0", "sentencepiece": "sentencepiece"}
        _missing_pkgs = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
        if _missing_pkgs:
            print(f"[bootstrap] Installing missing pip packages: {_missing_pkgs}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing_pkgs, check=False)

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[bootstrap] Attached GPU: {dev_name} ({vram_gb:.1f} GiB VRAM)")

    print("[bootstrap] Environment ready.")
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("[bootstrap] Local CPU testing mode enabled (RUN_SMOKE_CPU=True).")


In [ ]:
# 2. 20-Direction Random Orthogonal Hypersphere Null Generator on S^510
def generate_20_orthogonal_null_directions(
    d_ace: torch.Tensor,
    seed_base: int = 202704,
    device: str = "cpu"
) -> torch.Tensor:
    """
    Constructs 20 random unit vectors in S^510 strictly orthogonal to d_hat_ACE in R^512.
    Uses Gram-Schmidt orthogonalization: v_orth = v - <v, d_hat> d_hat.
    Asserts |<v_orth_unit, d_hat>| < 1e-6.
    """
    d_norm = d_ace / d_ace.norm(p=2).clamp(min=1e-12)
    dim = d_ace.shape[0]
    null_dirs = []
    
    for k in range(20):
        g = torch.Generator(device=device).manual_seed(seed_base + k)
        v = torch.randn(dim, generator=g, dtype=d_ace.dtype, device=device)
        proj = torch.dot(v, d_norm)
        v_orth = v - proj * d_norm
        v_unit = v_orth / v_orth.norm(p=2).clamp(min=1e-12)
        
        orth_err = abs(torch.dot(v_unit, d_norm).item())
        assert orth_err < 1e-6, f"Orthogonality assertion failed for null dir {k}: error={orth_err}"
        null_dirs.append(v_unit)
        
    return torch.stack(null_dirs)

def apply_ace_steering_hook(
    u: torch.Tensor,
    d_hat: torch.Tensor,
    alpha: float = 0.5,
    mode: str = "projection"
) -> torch.Tensor:
    """
    Applies ACE steering to self-conditioning feedback u of shape (B, L, d).
    Projection mode: u_steered = u - alpha * <u, d_hat> * d_hat
    """
    d_view = d_hat.view(1, 1, -1).to(dtype=u.dtype, device=u.device)
    if mode == "projection":
        proj_coeff = (u * d_view).sum(dim=-1, keepdim=True)
        return u - float(alpha) * proj_coeff * d_view
    elif mode == "subtraction":
        u_norm = u.norm(p=2, dim=-1, keepdim=True)
        return u - float(alpha) * u_norm * d_view
    else:
        raise ValueError(f"Unsupported ACE mode: {mode}")

In [ ]:
# 3. Published-Scale ACE Calibration Engine (4,096 Trajectories)
CALIBRATION_SEEDS = [301, 302, 303, 304, 305]  # 5 seeds x 820 = 4,100 trajectories
EVALUATION_SEEDS  = [201, 202, 203, 204, 205]  # Held-out evaluation seeds

def estimate_ace_direction_published(
    traj_mean_u: torch.Tensor,
    rep4_scores: np.ndarray,
    tail_quantile: float = 0.25
) -> Dict[str, Any]:
    """
    Computes d_hat_ACE by contrasting upper and lower rep-4 quartiles over calibration trajectories.
    """
    M_cal, d_lat = traj_mean_u.shape
    k_tail = max(1, int(round(M_cal * tail_quantile)))
    order = np.argsort(rep4_scores)
    
    low_idx = torch.as_tensor(order[:k_tail], dtype=torch.long, device=traj_mean_u.device)
    high_idx = torch.as_tensor(order[-k_tail:], dtype=torch.long, device=traj_mean_u.device)

    mu_low = traj_mean_u[low_idx].mean(dim=0)
    mu_high = traj_mean_u[high_idx].mean(dim=0)
    diff = mu_high - mu_low
    d_hat_ace = diff / diff.norm(p=2).clamp(min=1e-12)

    return {
        "d_hat_ace": d_hat_ace,
        "mu_high_norm": float(mu_high.norm(p=2).item()),
        "mu_low_norm": float(mu_low.norm(p=2).item()),
        "contrast_norm": float(diff.norm(p=2).item()),
        "mean_rep4_high_tail": float(np.mean(rep4_scores[order[-k_tail:]])),
        "mean_rep4_low_tail": float(np.mean(rep4_scores[order[:k_tail]])),
        "n_trajectories": M_cal
    }

def verify_calibration_cosine_stability(
    d_ace_4096: torch.Tensor,
    d_ace_500: torch.Tensor,
    threshold: float = 0.85
) -> float:
    """Computes cosine similarity between 4,096-sample and 500-sample ACE directions."""
    u1 = d_ace_4096 / d_ace_4096.norm(p=2)
    u2 = d_ace_500 / d_ace_500.norm(p=2)
    sim = float(abs(torch.dot(u1, u2).item()))
    print(f"[stability] Cosine stability |<d_4096, d_500>| = {sim:.4f} (Threshold: {threshold:.2f})")
    assert sim >= threshold, f"ACE direction stability check failed: {sim:.4f} < {threshold:.2f}"
    return sim

In [ ]:
# 4. Gamma-Compensated Matched ACE Calibration & Held-Out TOST
TARGET_BASELINE_PPL = 27.50  # Pair C3 baseline PPL
TARGET_BASELINE_H   = 5.190  # Pair C3 baseline H_ps
PPL_EQUIV_MARGIN    = 27.50 * 0.025  # +/- 0.6875
H_EQUIV_MARGIN      = 0.015          # +/- 0.015 nats

def solve_gamma_compensated_root(
    gamma_eval_points: List[float],
    ppl_values: List[float],
    target_ppl: float = TARGET_BASELINE_PPL
) -> float:
    """
    Fits a monotonic 1D spline / polynomial over calibration points to solve
    root gamma_comp* such that Gen-PPL(gamma_comp*) == target_ppl.
    """
    from scipy.interpolate import interp1d
    x = np.array(gamma_eval_points, dtype=float)
    y = np.array(ppl_values, dtype=float)
    order = np.argsort(x)
    x, y = x[order], y[order]
    
    # Monotonic inverse mapping
    f_inv = interp1d(y, x, kind="linear", fill_value="extrapolate")
    gamma_comp = float(f_inv(target_ppl))
    # Safety clamp to feasible back-noising range
    gamma_comp = float(np.clip(gamma_comp, 0.50, 1.50))
    print(f"[calibrate] Solved gamma_comp* = {gamma_comp:.4f} for target Gen-PPL={target_ppl:.2f}")
    return gamma_comp

def execute_matched_ace_tost(
    steered_records: List[Dict[str, Any]],
    baseline_records: List[Dict[str, Any]]
) -> Dict[str, Any]:
    """
    Executes formal paired Joint TOST on (Gen-PPL, H_ps) between:
    - Arm A: Unsteered baseline (SC=1.0, gamma*=1.651)
    - Arm B: Gamma-compensated Matched ACE (SC=4.0, gamma_comp*, alpha=0.5)
    Evaluates residual repetition reduction at the matched operating point.
    """
    try:
        import frontier_audit.tost as T
    except ImportError:
        import types
        from scipy import stats as _sp_stats
        class _TOSTRes:
            def __init__(self, diff, p_tost, p_two_sided_diff, is_equivalent):
                self.diff = diff
                self.p_tost = p_tost
                self.p_two_sided_diff = p_two_sided_diff
                self.is_equivalent = is_equivalent
            def to_dict(self):
                return self.__dict__
        def _tost_paired(a, b, low, high, alpha=0.05):
            d = np.asarray(b, dtype=np.float64) - np.asarray(a, dtype=np.float64)
            n = len(d)
            df = float(n - 1)
            md = float(np.mean(d))
            se = max(float(np.std(d, ddof=1)) / np.sqrt(n), 1e-15)
            p1 = float(1.0 - _sp_stats.t.cdf((md - low) / se, df=df))
            p2 = float(1.0 - _sp_stats.t.cdf((high - md) / se, df=df))
            p_tost = max(p1, p2)
            p_two = float(2.0 * (1.0 - _sp_stats.t.cdf(abs(md / se), df=df)))
            return _TOSTRes(md, p_tost, p_two, bool(p_tost < alpha))
        def _joint_tost(res_list):
            return {
                "p_joint": float(max(r.p_tost for r in res_list)),
                "is_equivalent": bool(all(r.is_equivalent for r in res_list)),
                "individual_results": [r.to_dict() for r in res_list],
            }
        def _holm_bonferroni(p_values, alpha=0.05):
            keys = list(p_values.keys())
            raw_p = np.array([p_values[k] for k in keys], dtype=np.float64)
            m = len(raw_p)
            sort_idx = np.argsort(raw_p)
            sorted_p = raw_p[sort_idx]
            adj_sorted = np.zeros(m, dtype=np.float64)
            run_max = 0.0
            for rank, p in enumerate(sorted_p):
                run_max = max(run_max, p * (m - rank))
                adj_sorted[rank] = min(1.0, run_max)
            adj_p = np.zeros(m, dtype=np.float64)
            adj_p[sort_idx] = adj_sorted
            return {
                "adjusted_p_values": {keys[i]: float(adj_p[i]) for i in range(m)},
                "significant": {keys[i]: bool(adj_p[i] < alpha) for i in range(m)},
                "m": m,
                "alpha": alpha,
            }
        def _hierarchical_gatekeeping(f1_p, f2_p, alpha=0.05):
            f1_h = _holm_bonferroni(f1_p, alpha=alpha)
            f1_pass = all(f1_h["significant"].values())
            if f1_pass:
                gate_status = "OPEN"
                f2_h = _holm_bonferroni(f2_p, alpha=alpha)
                f2_adj, f2_sig = f2_h["adjusted_p_values"], f2_h["significant"]
            else:
                gate_status = "CLOSED"
                f2_adj = {k: 1.0 for k in f2_p}
                f2_sig = {k: False for k in f2_p}
            return {
                "gate_status": gate_status,
                "family1": {
                    "name": "Family 1: Operating-Point Equivalence Gate",
                    "m": len(f1_p), "alpha": alpha, "raw_p_values": f1_p,
                    "adjusted_p_values": f1_h["adjusted_p_values"],
                    "significant": f1_h["significant"], "all_passed": f1_pass,
                },
                "family2": {
                    "name": "Family 2: Confirmatory Secondary Metrics",
                    "m": len(f2_p), "alpha": alpha, "raw_p_values": f2_p,
                    "adjusted_p_values": f2_adj, "significant": f2_sig,
                    "n_significant": sum(1 for v in f2_sig.values() if v),
                },
            }
        T = types.SimpleNamespace(
            tost_paired=_tost_paired,
            joint_tost=_joint_tost,
            holm_bonferroni=_holm_bonferroni,
            hierarchical_gatekeeping=_hierarchical_gatekeeping,
        )
    
    ppl_a = [r["gen_ppl"] for r in baseline_records]
    ppl_b = [r["gen_ppl"] for r in steered_records]
    h_a = [r["h_ps"] for r in baseline_records]
    h_b = [r["h_ps"] for r in steered_records]
    rep4_a = [r["rep4"] for r in baseline_records]
    rep4_b = [r["rep4"] for r in steered_records]
    
    res_ppl = T.tost_paired(ppl_a, ppl_b, -PPL_EQUIV_MARGIN, PPL_EQUIV_MARGIN)
    res_h = T.tost_paired(h_a, h_b, -H_EQUIV_MARGIN, H_EQUIV_MARGIN)
    joint = T.joint_tost([res_ppl, res_h])
    
    delta_rep4 = float(np.mean(rep4_b) - np.mean(rep4_a))
    rep4_diff_t = T.tost_paired(rep4_a, rep4_b, -0.01, 0.01)
    
    return {
        "p_joint_tost": joint["p_joint"],
        "p_ppl_tost": res_ppl.p_tost,
        "p_h_tost": res_h.p_tost,
        "is_operating_point_matched": (joint["p_joint"] < 0.05),
        "mean_baseline_ppl": float(np.mean(ppl_a)),
        "mean_steered_ppl": float(np.mean(ppl_b)),
        "mean_baseline_h": float(np.mean(h_a)),
        "mean_steered_h": float(np.mean(h_b)),
        "delta_rep4_matched": delta_rep4,
        "p_rep4_diff": rep4_diff_t.p_two_sided_diff
    }

In [ ]:
# 5. Execution Router (CPU Smoke / GPU Preflight / Production Matched ACE)
def run_smoke_cpu():
    """Runs fast CPU verification of 20 orthogonal directions, calibration, root-finding, and TOST."""
    print("[smoke] Running Notebook 21 CPU smoke test...")
    
    # 1. 20-direction orthogonal null verification
    d_dummy = torch.randn(512)
    d_dummy = d_dummy / d_dummy.norm(p=2)
    null_dirs = generate_20_orthogonal_null_directions(d_dummy, seed_base=202704, device="cpu")
    assert null_dirs.shape == (20, 512), f"Unexpected shape: {null_dirs.shape}"
    for k in range(20):
        assert abs(torch.dot(null_dirs[k], d_dummy).item()) < 1e-6
    
    # 2. Calibration verification
    rng = np.random.RandomState(42)
    u_mock = torch.randn(20, 512)
    rep4_mock = rng.uniform(0.05, 0.25, size=20)
    cal_res = estimate_ace_direction_published(u_mock, rep4_mock, tail_quantile=0.25)
    assert cal_res["d_hat_ace"].shape == (512,)
    
    # 3. Monotonic root-finding
    gammas = [0.70, 0.85, 1.00, 1.15]
    ppls = [29.5, 28.2, 27.1, 26.0]
    g_star = solve_gamma_compensated_root(gammas, ppls, target_ppl=27.50)
    assert 0.85 <= g_star <= 1.00, f"Unexpected root: {g_star}"
    
    # 4. Matched TOST
    base_records = [
        {"gen_ppl": float(27.5 + rng.normal(0, 0.1)), "h_ps": float(5.19 + rng.normal(0, 0.003)), "rep4": float(0.08 + rng.normal(0, 0.005))}
        for _ in range(5)
    ]
    steered_records = [
        {"gen_ppl": float(27.5 + rng.normal(0, 0.1)), "h_ps": float(5.19 + rng.normal(0, 0.003)), "rep4": float(0.016 + rng.normal(0, 0.005))}
        for _ in range(5)
    ]
    tost_res = execute_matched_ace_tost(steered_records, base_records)
    assert "p_joint_tost" in tost_res
    print(f"[smoke] Matched ACE TOST: p_joint={tost_res['p_joint_tost']:.4f}, delta_rep4={tost_res['delta_rep4_matched']:.4f}")
    print("[smoke] Notebook 21 CPU smoke test completed successfully.")

def run_preflight_gpu():
    """
    Tiny Real GPU Preflight Mode (~2 minutes on Colab A100):
    Loads real ELF-B, tests ACE projection hook during generation of M=8 samples,
    and scores perplexity with gpt2-large on CUDA.
    """
    print("="*70)
    print("[preflight-gpu] Launching Tiny Real Colab GPU Preflight on Matched ACE...")
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: GPU Preflight requires an attached CUDA GPU.")
    import elfx.load
    import elfx.sample
    import elfx.metrics
    import elfx.sweep
    
    t0 = time.time()
    # 1. Model Loading
    model_dict = elfx.sweep.get_model("ELF-B")
    print(f"[preflight-gpu] Loaded ELF-B model on CUDA ({time.time() - t0:.1f}s).")
    
    # 2. Dummy ACE Direction Hook
    d_test = torch.randn(512, device="cuda")
    d_test = d_test / d_test.norm(p=2)
    
    def _preflight_hook(i, t, t_next, t_eff, z, x_pred):
        x_steered = apply_ace_steering_hook(x_pred, d_test, alpha=0.5, mode="projection")
        return z, x_steered

    # 3. Rollout M=8 samples at K=4 steps with ACE hook
    print("[preflight-gpu] Rolling out M=8 sequences with ACE hook on CUDA...")
    gen_out = elfx.sample.generate(
        model=model_dict["model"],
        config=model_dict["config"],
        tokenizer=model_dict["tokenizer"],
        n=8,
        seed=201,
        steps=4,
        method="sde",
        gamma=1.0,
        sc_cfg=4.0,
        batch_size=8,
        hook=_preflight_hook
    )
    texts = gen_out["texts"]
    assert len(texts) == 8
    print(f"[preflight-gpu] Generated 8 samples in {time.time() - t0:.2f}s. Sample preview: {texts[0][:80]!r}...")
    
    # 4. Scoring
    sc = elfx.metrics.score_gpt2(texts, batch_size=4)
    ppl = elfx.metrics.corpus_ppl(sc["nll"], sc["count"])
    assert ppl > 1.0
    print(f"[preflight-gpu] Verified Gen-PPL = {ppl:.2f}. Matched ACE hook verified on GPU!")
    print("="*70)

def run_production_matched_ace():
    """Full Production Matched ACE & 20-Direction Null Experiment."""
    if not torch.cuda.is_available():
        raise ColabEngineeringError("FAIL-CLOSED: Production matched ACE requires a CUDA GPU.")
    print("="*70)
    print("[production] Beginning Full Production Matched ACE Execution...")
    print("="*70)
    
    out_dir = PROJECT_ROOT / "runs" / "p21_matched_ace"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    import elfx.load
    import elfx.sample
    import elfx.metrics
    import elfx.refs
    import elfx.sweep
    from elfx import non_identifiability as ni
    
    model_dict = elfx.sweep.get_model("ELF-B")
    
    # --- PHASE 1: PUBLISHED-SCALE CALIBRATION (4,096 Trajectories across Seeds 301..305) ---
    cal_file = out_dir / "ace_calibrated_directions_4096.pt"
    if cal_file.is_file():
        print(f"[production] Loading cached 4,096-trajectory ACE direction from {cal_file}...")
        cal_data = torch.load(cal_file, map_location="cuda")
        d_hat_ace = cal_data["d_hat_ace"]
    else:
        print("[production] Running 4,096-trajectory calibration on seeds 301..305...")
        u_list, rep4_list = [], []
        _batch_u = []
        _state = {"u_sum": None}
        
        def _cal_record_hook(eager_model, encoder, tokenizer, config, steps=32):
            def _hook(i, t, t_next, t_eff, z, x_pred):
                u_mean = x_pred.detach().float().mean(dim=1)
                if i == 0 or _state["u_sum"] is None:
                    _state["u_sum"] = u_mean
                else:
                    _state["u_sum"] = _state["u_sum"] + u_mean
                if i == steps - 2:
                    _batch_u.append((_state["u_sum"] / float(steps - 1)).cpu())
                    _state["u_sum"] = None
                return z, x_pred
            return _hook
            
        elfx.sweep.HOOKS["ace_record"] = _cal_record_hook
        for cseed in CALIBRATION_SEEDS:
            _batch_u.clear()
            cal_cfg = {
                "run_id": f"p21_cal_seed{cseed}",
                "model": "ELF-B",
                "steps": 32,
                "method": "sde",
                "gamma": 1.0,
                "sc_cfg": 4.0,
                "n": 820,
                "seed": int(cseed),
                "mauve": False,
                "hook": "ace_record",
                "hook_kwargs": {"steps": 32}
            }
            elfx.sweep.run_config(cal_cfg, verbose=True)
            u_seed = torch.cat(_batch_u, dim=0)[:820]
            ps = np.load(PROJECT_ROOT / "runs" / f"p21_cal_seed{cseed}" / "per_sample.npz")
            rep4_seed = ps["rep4"][:820]
            u_list.append(u_seed)
            rep4_list.append(rep4_seed)
            
        all_u = torch.cat(u_list, dim=0).cuda()
        all_rep4 = np.concatenate(rep4_list, axis=0)
        cal_res = estimate_ace_direction_published(all_u, all_rep4, tail_quantile=0.25)
        d_hat_ace = cal_res["d_hat_ace"]
        torch.save(cal_res, cal_file)
        print(f"[production] Saved 4,096-trajectory d_hat_ACE to {cal_file}")

    # Stability check against NB09 500-sample direction if present
    p9_cal = PROJECT_ROOT / "runs" / "p9_ace_intervention" / "ace_calibrated_directions.pt"
    if p9_cal.is_file():
        p9_data = torch.load(p9_cal, map_location="cuda")
        verify_calibration_cosine_stability(d_hat_ace, p9_data["d_hat_ace"], threshold=0.85)

    # --- PHASE 2: 20-DIRECTION ORTHOGONAL HYPERSPHERE NULL ---
    print("\n[production] Evaluating 20 random orthogonal null directions on S^510...")
    null_eval_path = out_dir / "ace_null_directions_evaluated.json"
    null_directions = generate_20_orthogonal_null_directions(d_hat_ace, seed_base=202704, device="cuda")
    null_records = []
    
    for k in range(20):
        v_null = null_directions[k]
        
        def _null_hook(i, t, t_next, t_eff, z, x_pred):
            return z, apply_ace_steering_hook(x_pred, v_null, alpha=0.5, mode="projection")
            
        k_rep4s = []
        for eseed in EVALUATION_SEEDS:
            gen_out = elfx.sample.generate(
                model=model_dict["model"], config=model_dict["config"], tokenizer=model_dict["tokenizer"],
                n=500, seed=eseed, steps=32, method="sde", gamma=1.0, sc_cfg=4.0, batch_size=64,
                hook=_null_hook
            )
            sc = elfx.metrics.score_gpt2(gen_out["texts"], batch_size=32)
            k_rep4s.append(float(elfx.metrics.rep_n(gen_out["texts"], 4)))
        null_records.append({"null_idx": k, "mean_rep4": float(np.mean(k_rep4s))})
        print(f"  Null Dir {k:02d}: rep-4 = {np.mean(k_rep4s):.4f}")
        
    with open(null_eval_path, "w", encoding="utf-8") as f:
        json.dump(null_records, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

    # --- PHASE 3: GAMMA-COMPENSATED MATCHED ACE ROOT-FINDING ---
    print("\n[production] Calibrating gamma_comp* on calibration seeds 301..302...")
    cal_grid_gammas = [0.70, 0.85, 1.00, 1.15]
    grid_ppls = []
    
    def _ace_proj_hook(i, t, t_next, t_eff, z, x_pred):
        return z, apply_ace_steering_hook(x_pred, d_hat_ace, alpha=0.5, mode="projection")
        
    for g_val in cal_grid_gammas:
        seed_ppls = []
        for cseed in [301, 302]:
            gen_out = elfx.sample.generate(
                model=model_dict["model"], config=model_dict["config"], tokenizer=model_dict["tokenizer"],
                n=500, seed=cseed, steps=32, method="sde", gamma=g_val, sc_cfg=4.0, batch_size=64,
                hook=_ace_proj_hook
            )
            sc = elfx.metrics.score_gpt2(gen_out["texts"], batch_size=32)
            seed_ppls.append(float(elfx.metrics.corpus_ppl(sc["nll"], sc["count"])))
        grid_ppls.append(float(np.mean(seed_ppls)))
        print(f"  Grid gamma={g_val:.2f} -> Gen-PPL={np.mean(seed_ppls):.2f}")
        
    gamma_comp_star = solve_gamma_compensated_root(cal_grid_gammas, grid_ppls, target_ppl=27.50)
    
    # --- PHASE 4: HELD-OUT EVALUATION & TOST ON SEEDS 201..205 ---
    print(f"\n[production] Executing held-out matched evaluation on seeds 201..205 at gamma_comp*={gamma_comp_star:.4f}...")
    matched_records = []
    baseline_records = []
    
    for eseed in EVALUATION_SEEDS:
        # Load or generate baseline Pair C3 Arm A (SC=1.0, gamma*=1.651)
        base_summ = PROJECT_ROOT / "runs" / f"p4_elfb_s32_g1.651_sc1.0_seed{eseed}" / "summary.json"
        if base_summ.is_file():
            with open(base_summ, "r", encoding="utf-8") as f:
                raw_b = json.load(f)
            baseline_records.append({
                "seed": eseed,
                "gen_ppl": float(raw_b.get("gen_ppl", raw_b.get("ppl"))),
                "h_ps": float(raw_b.get("h_ps", raw_b.get("entropy", raw_b.get("token_entropy")))),
                "rep4": float(raw_b["rep4"]),
            })
        else:
            base_out = elfx.sample.generate(
                model=model_dict["model"], config=model_dict["config"], tokenizer=model_dict["tokenizer"],
                n=1000, seed=eseed, steps=32, method="sde", gamma=1.651, sc_cfg=1.0, batch_size=64
            )
            base_sc = elfx.metrics.score_gpt2(base_out["texts"], batch_size=32)
            baseline_records.append({
                "seed": eseed,
                "gen_ppl": float(elfx.metrics.corpus_ppl(base_sc["nll"], base_sc["count"])),
                "h_ps": float(np.nanmean(base_sc["entropy"])),
                "rep4": float(elfx.metrics.rep_n(base_out["texts"], 4)),
            })
                
        # Generate matched steered arm
        gen_out = elfx.sample.generate(
            model=model_dict["model"], config=model_dict["config"], tokenizer=model_dict["tokenizer"],
            n=1000, seed=eseed, steps=32, method="sde", gamma=gamma_comp_star, sc_cfg=4.0, batch_size=64,
            hook=_ace_proj_hook
        )
        sc = elfx.metrics.score_gpt2(gen_out["texts"], batch_size=32)
        ppl = float(elfx.metrics.corpus_ppl(sc["nll"], sc["count"]))
        h_ps = float(np.nanmean(sc["entropy"]))
        rep4 = float(elfx.metrics.rep_n(gen_out["texts"], 4))
        
        matched_records.append({
            "seed": eseed,
            "gamma": gamma_comp_star,
            "sc_cfg": 4.0,
            "gen_ppl": ppl,
            "h_ps": h_ps,
            "rep4": rep4
        })
        print(f"  Seed {eseed}: Steered Gen-PPL={ppl:.2f}, H_ps={h_ps:.4f}, rep-4={rep4:.4f}")
        
    tost_out = execute_matched_ace_tost(matched_records, baseline_records)
    print(f"\n[production] Matched ACE Joint TOST: p_joint = {tost_out['p_joint_tost']:.4f}")
    print(f"  Delta rep-4 at matched operating point: {tost_out['delta_rep4_matched']:+.4f} (p = {tost_out['p_rep4_diff']:.4f})")

    # --- PHASE 5: SAVE MANIFEST & SUMMARY ---
    summary_out = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "gamma_comp_star": gamma_comp_star,
        "calibration_grid": {"gammas": cal_grid_gammas, "ppls": grid_ppls},
        "tost_evaluation": tost_out,
        "null_distribution": null_records
    }
    summary_path = out_dir / "matched_ace_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_out, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
        
    manifest = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "summary_sha256": hashlib.sha256(summary_path.read_bytes()).hexdigest(),
        "cal_directions_sha256": hashlib.sha256(cal_file.read_bytes()).hexdigest(),
        "null_eval_sha256": hashlib.sha256(null_eval_path.read_bytes()).hexdigest()
    }
    with open(out_dir / "matched_ace_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"\n[production] Matched ACE intervention completed successfully!")
    print(f"[production] Artifacts archived to {out_dir}")

if RUN_SMOKE_CPU:
    run_smoke_cpu()
elif RUN_PREFLIGHT_GPU:
    run_preflight_gpu()
else:
    run_production_matched_ace()